<a href="https://colab.research.google.com/github/aishwaryamaheshb-coder/5L-AI-Lab/blob/main/1WA24CS026_Week2_DFS_IDS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
INITIAL = (1, 2, 3,
           0, 4, 6,
           7, 5, 8)
GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)
MOVES = [("Up", -1, 0), ("Down", 1, 0), ("Left", 0, -1), ("Right", 0, 1)]
def get_neighbors(state):
    b = state.index(0)
    r, c = divmod(b, 3)
    result = []
    for name, dr, dc in MOVES:
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            n = nr * 3 + nc
            new = list(state)
            new[b], new[n] = new[n], new[b]
            result.append((name, tuple(new)))
    return result
def is_solvable(state, goal=GOAL):
    def inv(s):
        t = [x for x in s if x]
        return sum(1 for i in range(8) for j in range(i + 1, 8) if t[i] > t[j])
    return inv(state) % 2 == inv(goal) % 2
def show(state, indent="   "):
    for r in range(3):
        print(indent + " ".join(str(x) if x else "_" for x in state[3 * r:3 * r + 3]))
#Depth First Search
def dfs(start, goal, max_depth=None):
    stack = [(start, [(None, start)])]
    visited = set()
    expanded = 0
    while stack:
        state, path = stack.pop()
        if state in visited:
            continue
        visited.add(state)
        expanded += 1
        if state == goal:
            return path, expanded
        if max_depth is not None and len(path) - 1 >= max_depth:
            continue
        for move, nxt in reversed(get_neighbors(state)):
            if nxt not in visited:
                stack.append((nxt, path + [(move, nxt)]))
    return None, expanded

#Iterative Deepening Search
def depth_limited_search(state, goal, limit, path, on_path, counter):
    """Recursive DLS. Returns path, 'cutoff' or None (failure)."""
    counter[0] += 1
    if state == goal:
        return path
    if limit == 0:
        return "cutoff"
    cutoff_occurred = False
    for move, nxt in get_neighbors(state):
        if nxt in on_path:
            continue
        on_path.add(nxt)
        result = depth_limited_search(nxt, goal, limit - 1,
                                      path + [(move, nxt)], on_path, counter)
        on_path.remove(nxt)
        if result == "cutoff":
            cutoff_occurred = True
        elif result is not None:
            return result
    return "cutoff" if cutoff_occurred else None


def ids(start, goal, max_limit=50):
    total = [0]
    for depth in range(max_limit + 1):
        counter = [0]
        result = depth_limited_search(start, goal, depth, [(None, start)],
                                      {start}, counter)
        total[0] += counter[0]
        print(f"  depth limit = {depth:2d}  -> nodes visited: {counter[0]:6d}  "
              f"{'GOAL FOUND' if result not in (None, 'cutoff') else 'cutoff'}")
        if result not in (None, "cutoff"):
            return result, total[0]
    return None, total[0]


def print_solution(path, show_boards=True):
    print(f"\nSolution length: {len(path) - 1} moves")
    for i, (move, st) in enumerate(path):
        if len(path) > 25 and 3 < i < len(path) - 3:
            if i == 4:
                print("   ... (middle steps omitted) ...")
            continue
        print(f"Step {i}: {move or 'Start'}")
        if show_boards:
            show(st)


if __name__ == "__main__":
    if not is_solvable(INITIAL):
        raise SystemExit("Puzzle is not solvable")

    print("=" * 50)
    print("(i) DEPTH FIRST SEARCH")
    print("=" * 50)
    path, expanded = dfs(INITIAL, GOAL)
    print_solution(path)
    print(f"Nodes expanded: {expanded}")

    print("\n" + "=" * 50)
    print("(ii) ITERATIVE DEEPENING SEARCH")
    print("=" * 50)
    path, total = ids(INITIAL, GOAL)
    print_solution(path)
    print(f"Total nodes visited over all iterations: {total}")

(i) DEPTH FIRST SEARCH

Solution length: 833 moves
Step 0: Start
   1 2 3
   _ 4 6
   7 5 8
Step 1: Up
   _ 2 3
   1 4 6
   7 5 8
Step 2: Right
   2 _ 3
   1 4 6
   7 5 8
Step 3: Down
   2 4 3
   1 _ 6
   7 5 8
   ... (middle steps omitted) ...
Step 831: Down
   1 2 3
   4 _ 6
   7 5 8
Step 832: Down
   1 2 3
   4 5 6
   7 _ 8
Step 833: Right
   1 2 3
   4 5 6
   7 8 _
Nodes expanded: 853

(ii) ITERATIVE DEEPENING SEARCH
  depth limit =  0  -> nodes visited:      1  cutoff
  depth limit =  1  -> nodes visited:      4  cutoff
  depth limit =  2  -> nodes visited:      9  cutoff
  depth limit =  3  -> nodes visited:     16  GOAL FOUND

Solution length: 3 moves
Step 0: Start
   1 2 3
   _ 4 6
   7 5 8
Step 1: Right
   1 2 3
   4 _ 6
   7 5 8
Step 2: Down
   1 2 3
   4 5 6
   7 _ 8
Step 3: Right
   1 2 3
   4 5 6
   7 8 _
Total nodes visited over all iterations: 30
